**Навигация по уроку**
1. [Архитектура автокодировщиков](https://colab.research.google.com/drive/1su5Dlu7SlFzmWlEw_Ix5q5Ps6PicwRND)
2. [Обнаружение выбросов и аномалий](https://colab.research.google.com/drive/1oLBwktGpPIqt8YLygBR1H45rdqw4sZdh)
3. [Удаление водяных знаков с помощью сверточных автокодировщиков](https://colab.research.google.com/drive/1L00DEOKX5yuHlHY0VhxBo3sGSWsjnJsB)
4. Домашняя работа

В домашней работе вам необходимо выполнить одно из трёх заданий на выбор:


**Задание 1. На 3 балла:**

1. Ваша задача научить автокодировщик очищать шум на изображениях.

2. Используйте датасет MNIST Fashion:

```python
 keras.datasets.fashion_mnist.load_data()
```

3. В процессе обучения, на вход необходимо подавать изображения с шумом, на выход без шума.

4. Сгенерируйте шум:

```python
# генерируем тензор случайных значения равный по форме наших данным
train_noise = np.random.standard_normal(y_train.shape)/30 + .5
test_noise = np.random.standard_normal(y_test.shape)/30 + .5
```

5. Создайте зашумленные изображения:

```python
X_train = y_train + train_noise
X_test = y_test + test_noise
```
6. Подумайте над архитектурой автокодировщика.

7. Обучите модель и сделайте предсказания на контрольной выборке.

8. Продемонстрируйте работу модели путём вывода не менее 5 примеров из тестовой выборки, содержащих: исходное изображение, зашумленное изображение, предсказанное изображение и изображение шума (полученное путём вычитания предсказанного изображения из зашумленного изображения).

**Задание 2. На 4 балла:**
1. Используйте  изображения без водяных знаков из 3-й части урока (https://storage.yandexcloud.net/academy.ai/watermarked.zip)

2. Придумайте механизм нанесения водяных знаков (может быть обычный текст) на изображения. Нанесите водяные знаки на входные изображения.

3. Подумайте над архитектурой автокодировщика.

4. Размер картинок, число образцов для обучения выбирайте, исходя из ограничения среды выполнения, например,
ограничения ОЗУ бесплатного колаба.

5. Обучите модель и сделайте предсказания на контрольной выборке.

6. Продемонстрируйте работу модели путём вывода не менее 10 примеров из тестовой выборки, содержащих: исходное изображение, изображение с водяными знаками, предсказанное изображение и изображение шума (полученное путём вычитания предсказанного изображения из изображения).

**Задание 3. На 5 баллов необходимо выполнить Задание 2 со следующими улучшениями:**
1. Применить аугментацию к изображениям, не менее 5-ти модификаций (трансформаций) картинок.
2. Проведите пошаговое обучение (не менее 3-х шагов). Обучайте модель в несколько итерации: загрузили часть данных, обучили на них, почистили ОЗУ, при необходимости скорректировали параметры обучения. Это позволит использовать оперативную память более эффективно и достигнуть лучшего обучения модели.

In [ ]:
!pip install keras

In [ ]:
import warnings
import matplotlib.image as mpimg

warnings.filterwarnings(
    'ignore',
    category=UserWarning,
    module='matplotlib.image',
    message='Clipping input data.*'
)

In [ ]:
# Импорт библиотек
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import random
import string
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, BatchNormalization, PReLU, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, Callback, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import gc
import time
import psutil
import math

In [ ]:
# --- Включение Mixed Precision ---
gpu_available = tf.config.list_physical_devices('GPU')
if gpu_available:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("Политика Mixed Precision установлена на 'mixed_float16'.")
    except Exception as e:
        print(f"Не удалось установить политику Mixed Precision: {e}")
        print("Mixed Precision не включена.")
else:
    print("GPU не обнаружен или несовместим. Mixed Precision не включена.")

In [ ]:
# Глобальные параметры для конфигурации
# Тут собраны основные настройки, которые определяют размер изображений,
# сколько данных мы используем, как будем учить модель и куда сохранять результаты.

IMG_WIDTH = 128 # Ширина изображения в пикселях. Модель будет работать с таким размером.
IMG_HEIGHT = 128 # Высота изображения в пикселях. Тоже фиксированный размер для модели.
IMG_SIZE = (IMG_WIDTH, IMG_HEIGHT) # Удобный кортеж для указания размера (ширина, высота).

# Сколько всего примеров из датасета мы планируем использовать
# для тренировки, валидации и финального тестирования соответственно.
# Выбираются случайным образом из общего набора.
TRAIN_SAMPLE_SIZE = 8000 # Количество картинок для тренировки.
VALID_SAMPLE_SIZE = 1500 # Количество картинок для валидации (для оценки в процессе обучения).
TEST_SAMPLE_SIZE = 500 # Количество картинок для финальной оценки (модель их не видела).

BATCH_SIZE = 32 # Размер мини-батча. Столько картинок модель обрабатывает за один шаг градиентного спуска.
# Зависит от видеопамяти: чем больше батч, тем быстрее может быть обучение, но нужно больше памяти.

INITIAL_LR = 0.0001 # Начальная скорость обучения. Насколько сильно веса модели будут меняться на каждой итерации.

# --- Параметры для пошагового обучения (это для эффективного использования памяти) ---
# Мы делим всю тренировку на большие "шаги". На каждом шаге грузим только часть данных.
NUM_TRAINING_STEPS = 50 # Сколько таких больших шагов тренировки у нас будет всего.
EPOCHS_PER_STEP = 20 # Сколько раз модель пройдет по ВСЕМ данным, загруженным для текущего шага.
# То есть, на каждом шаге мы обучаемся на 160 (или сколько там в чанке) картинках в течение 20 эпох.
# Общее число эффективных эпох примерно = NUM_TRAINING_STEPS * EPOCHS_PER_STEP.

# Сколько валидационных примеров мы будем использовать для оценки модели
# после каждого шага тренировки, чтобы видеть прогресс.
MONITOR_VALIDATION_SAMPLES = 20

# --- Параметры для ручного управления обучением (пошагово) ---
# Это логика для автоматической коррекции скорости обучения и остановки.
# Мы смотрим на валидационные метрики после каждого шага.
LR_REDUCTION_FACTOR = 0.5 # Во сколько раз снижать скорость обучения, если метрика не улучшается.
LR_REDUCTION_PATIENCE_STEPS = 5 # Сколько шагов ждать без улучшения валидационной метрики, прежде чем снизить LR.
EARLY_STOPPING_PATIENCE_STEPS = 15 # Сколько шагов ждать без улучшения, прежде чем полностью остановить обучение.

MIN_LR = 0.000001 # Минимальная скорость обучения. Ниже этого значения LR не опустится.

MODEL_SAVE_PATH = 'best_denoiser_model_weights.weights.h5' # Путь к файлу, куда будут сохраняться веса модели.
# Мы сохраняем веса лучшей модели по валидационной метрике (SSIM).

In [ ]:
# Метрики
def psnr_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true = tf.clip_by_value(y_true, 0.0, 1.0)
    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)

    psnr_values = tf.image.psnr(y_true, y_pred, max_val=1.0)

    return tf.reduce_mean(psnr_values)

def ssim_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true = tf.clip_by_value(y_true, 0.0, 1.0)
    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)

    if len(y_true.shape) == 3:
        y_true = tf.expand_dims(y_true, axis=0)
    if len(y_pred.shape) == 3:
        y_pred = tf.expand_dims(y_pred, axis=0)

    if y_true.shape[0] != y_pred.shape[0]:
         print(f"Warning: Shape mismatch for SSIM metric batch size: {y_true.shape} vs {y_pred.shape}")
         min_batch = min(y_true.shape[0], y_pred.shape[0])
         if min_batch == 0:
             return tf.constant(np.nan, dtype=tf.float32)
         y_true = y_true[:min_batch]
         y_pred = y_pred[:min_batch]

    ssim_values = tf.image.ssim(y_true, y_pred, max_val=1.0)
    return tf.reduce_mean(ssim_values)

In [ ]:
# Водяной знак
# Эта функция накладывает случайный текстовый водяной знак на изображение.
# Используется для создания обучающих пар.
# На вход ожидает изображение в формате NumPy (float32, [0, 1], HxWx3).
# Возвращает изображение с наложенным водяным знаком в том же формате.
def add_watermark(image, is_debug_simple=False):
    img_uint8 = (image * 255).astype(np.uint8).copy()

    h_img, w_img = img_uint8.shape[:2]

    overlay = img_uint8.copy()

    if is_debug_simple:
        text = "TEST WM"
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = min(h_img, w_img) / 300.0
        thickness = max(1, int(font_scale * 1.8))
        text_color = (200, 200, 200)
        shadow_color = (50, 50, 50)
        alpha = 0.5

        (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
        text_x = (w_img - tw) // 2 + random.randint(-w_img//20, w_img//20)
        text_y = (h_img + th) // 2 + random.randint(-h_img//20, h_img//20) + baseline

        text_x = np.clip(text_x, 0, max(0, w_img - tw))
        text_y = np.clip(text_y, th + baseline, max(th + baseline, h_img - 5))


        shadow_offset_x = max(1, int(font_scale / 2))
        shadow_offset_y = max(1, int(font_scale / 2))

        cv2.putText(overlay, text, (text_x + shadow_offset_x, text_y + shadow_offset_y), font, font_scale, shadow_color, thickness + max(1, thickness//3), cv2.LINE_AA)
        cv2.putText(overlay, text, (text_x, text_y), font, font_scale, text_color, thickness, cv2.LINE_AA)

        img_uint8 = cv2.addWeighted(overlay, alpha, img_uint8, 1 - alpha, 0)

    else:
        text_length = random.randint(8, 25)
        text = ''.join(random.choice(string.ascii_uppercase + string.digits + " .,!?" * 5) for _ in range(text_length))

        fonts = [cv2.FONT_HERSHEY_SIMPLEX, cv2.FONT_HERSHEY_PLAIN, cv2.FONT_HERSHEY_DUPLEX,
                 cv2.FONT_HERSHEY_COMPLEX, cv2.FONT_HERSHEY_TRIPLEX, cv2.FONT_HERSHEY_SCRIPT_SIMPLEX,
                 cv2.FONT_HERSHEY_SCRIPT_COMPLEX, cv2.FONT_ITALIC]
        font = random.choice(fonts)

        alpha = random.uniform(0.3, 0.9)

        current_font_scale = min(h_img, w_img) / random.uniform(80.0, 300.0)
        current_thickness = max(1, int(current_font_scale * random.uniform(1.0, 3.5)))

        max_fit_attempts = 30
        attempts = 0
        text_fits = False
        text_x, text_y = -1, -1

        while attempts < max_fit_attempts:
            (tw, th), baseline = cv2.getTextSize(text, font, current_font_scale, current_thickness)

            min_x = 0
            max_x = w_img - tw
            min_y = th + baseline
            max_y = h_img - 5

            if max_x >= min_x and max_y >= min_y:
                 text_fits = True
                 text_x = random.randint(min_x, max_x)
                 text_y = random.randint(min_y, max_y)
                 break
            else:

                 current_font_scale *= random.uniform(0.7, 0.95)
                 current_thickness = max(1, int(current_font_scale * random.uniform(1.0, 3.5)))
                 if current_font_scale < 0.1 or current_thickness < 1:
                     break

            attempts += 1

        if not text_fits:
             return image

        shadow_offset_factor = random.uniform(0.2, 1.2) # Random offset magnitude
        shadow_offset_x = max(1, int(current_thickness * shadow_offset_factor))
        shadow_offset_y = max(1, int(current_thickness * shadow_offset_factor))

        shadow_draw_x = text_x + shadow_offset_x
        shadow_draw_y = text_y + shadow_offset_y

        color_type = random.choice(['gray', 'light_color', 'dark_color', 'random_hue'])
        if color_type == 'gray':
             base_color_val = random.randint(80, 240)
             text_color = (base_color_val, base_color_val, base_color_val)
             shadow_offset_val = random.randint(20, 80)
             shadow_color = (max(0, base_color_val - shadow_offset_val), max(0, base_color_val - shadow_offset_val), max(0, base_color_val - shadow_offset_val))
             # Draw text and shadow now that colors are defined
             cv2.putText(overlay, text, (text_x, text_y), font, current_font_scale, text_color, current_thickness, cv2.LINE_AA)

        elif color_type == 'light_color': # light_color
             text_color = (random.randint(150, 255), random.randint(150, 255), random.randint(150, 255))
             shadow_offset_val = random.randint(30, 80)
             shadow_color = (max(0, text_color[0] - shadow_offset_val), max(0, text_color[1] - shadow_offset_val), max(0, text_color[2] - shadow_offset_val))
             cv2.putText(overlay, text, (shadow_draw_x, shadow_draw_y), font, current_font_scale, shadow_color, current_thickness + max(1, current_thickness//3), cv2.LINE_AA)
             cv2.putText(overlay, text, (text_x, text_y), font, current_font_scale, text_color, current_thickness, cv2.LINE_AA)

        elif color_type == 'dark_color':
             text_color = (random.randint(0, 100), random.randint(0, 100), random.randint(0, 100))
             shadow_offset_val = random.randint(30, 80)
             shadow_color = (min(255, text_color[0] + shadow_offset_val), min(255, text_color[1] + shadow_offset_val), min(255, text_color[2] + shadow_offset_val))
             cv2.putText(overlay, text, (shadow_draw_x, shadow_draw_y), font, current_font_scale, shadow_color, current_thickness + max(1, current_thickness//3), cv2.LINE_AA)
             cv2.putText(overlay, text, (text_x, text_y), font, current_font_scale, text_color, current_thickness, cv2.LINE_AA)

        else:
             hue = random.randint(0, 179)
             saturation = random.randint(50, 200)
             value = random.randint(150, 255)
             text_color_hsv = np.array([[[hue, saturation, value]]], dtype=np.uint8)
             text_color_bgr = cv2.cvtColor(text_color_hsv, cv2.COLOR_HSV2BGR)[0][0].tolist()
             text_color = (int(text_color_bgr[0]), int(text_color_bgr[1]), int(text_color_bgr[2]))

             shadow_value = max(0, value - random.randint(50, 150))
             shadow_saturation = min(255, saturation + random.randint(0, 50)) #
             shadow_color_hsv = np.array([[[hue, shadow_saturation, shadow_value]]], dtype=np.uint8)
             shadow_color_bgr = cv2.cvtColor(shadow_color_hsv, cv2.COLOR_HSV2BGR)[0][0].tolist()
             shadow_color = (int(shadow_color_bgr[0]), int(shadow_color_bgr[1]), int(shadow_color_bgr[2]))
             cv2.putText(overlay, text, (shadow_draw_x, shadow_draw_y), font, current_font_scale, shadow_color, current_thickness + max(1, current_thickness//3), cv2.LINE_AA)
             cv2.putText(overlay, text, (text_x, text_y), font, current_font_scale, text_color, current_thickness, cv2.LINE_AA)

        img_uint8 = cv2.addWeighted(overlay, alpha, img_uint8, 1 - alpha, 0)

    watermarked_img_float = img_uint8.astype(np.float32) / 255.0
    return watermarked_img_float

In [ ]:
def load_and_preprocess_chunk(filenames, image_dir, img_size, is_train=True, datagen=None):
    images_clean = []
    images_watermarked = []

    for filename in filenames:
        img_path = os.path.join(image_dir, filename)
        img = cv2.imread(img_path)
        if img is None:
            continue
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4:
             img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)


        img = cv2.resize(img, img_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img_uint8 = img.astype(np.uint8)

        img_uint8_post_aug = img_uint8.copy()

        if is_train and datagen:
             augmented_img_uint8 = datagen.random_transform(img_uint8_post_aug)
             if augmented_img_uint8.ndim == 2:
                  augmented_img_uint8 = cv2.cvtColor(augmented_img_uint8, cv2.COLOR_GRAY2RGB)
             img_uint8_post_aug = augmented_img_uint8


        img_clean_processed_post_aug = img_uint8_post_aug.astype(np.float32) / 255.0

        img_watermarked_processed = add_watermark(img_clean_processed_post_aug, is_debug_simple=False)


        images_clean.append(img_clean_processed_post_aug)
        images_watermarked.append(img_watermarked_processed)


    if not images_clean:
        return np.zeros((0, *img_size, 3), dtype=np.float32), np.zeros((0, *img_size, 3), dtype=np.float32)

    return np.array(images_watermarked, dtype=np.float32), np.array(images_clean, dtype=np.float32)


In [ ]:
def residual_block(x, filters):
    shortcut = x
    if shortcut.shape[-1] is not None and shortcut.shape[-1] != filters:
         shortcut = Conv2D(filters, (1,1), padding='same', kernel_initializer='he_normal', dtype=x.dtype)(shortcut)
         shortcut = BatchNormalization()(shortcut)

    x = Conv2D(filters, (3,3), padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = PReLU(shared_axes=[1,2])(x)

    x = Conv2D(filters, (3,3), padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)

    x = Add()([shortcut, x])

    x = PReLU(shared_axes=[1,2])(x)
    return x

In [ ]:
# Архитектура модели, попросил нейронку объянить более подробнее, потмоу что не совсем в началае понял как это происходит

# Это определение нашей нейронной сети. Мы используем структуру, похожую на U-Net,
# которая хорошо подходит для задач типа "изображение в изображение" (как удаление водяных знаков).
# Идея U-Net: сначала "сжимаем" изображение (энкодер) для извлечения признаков,
# а затем "расширяем" (декодер) для восстановления изображения,
# при этом используя "связи пропуска" (skip connections) для сохранения мелких деталей.

def build_model_no_se_predict_clean_boosted_filters():
    print("Используется архитектура модели из вашего скрипта. Убедитесь, что она верна.")

    # Входной слой модели. Указываем размер изображения и количество каналов (3 для RGB),
    # а также тип данных, который ожидаем (float32).
    inputs = Input((*IMG_SIZE, 3), dtype=tf.float32)

    # --- Энкодер (Сжимающая часть) ---
    # Энкодер постепенно уменьшает пространственное разрешение изображения (ширину и высоту),
    # но увеличивает количество "каналов" или фильтров, чтобы захватить более сложные признаки.

    # Блок 1 энкодера: Применяем residual_block (свертки + нормализация + активация),
    # затем Max Pooling для уменьшения размера в 2 раза.
    e1 = residual_block(inputs, 48) # Обработка входного изображения первым блоком.
    p1 = MaxPooling2D((2,2))(e1) # Уменьшаем размер изображения в 2 раза (например, 128x128 -> 64x64).

    # Блок 2 энкодера: Обрабатываем выход предыдущего пулинга, увеличиваем количество фильтров,
    # снова уменьшаем размер.
    e2 = residual_block(p1, 96) # Обработка данных после первого пулинга.
    p2 = MaxPooling2D((2,2))(e2) # Уменьшаем размер (64x64 -> 32x32).

    # Блок 3 энкодера: Еще один шаг обработки и уменьшения размера.
    e3 = residual_block(p2, 192) # Обработка данных после второго пулинга.
    p3 = MaxPooling2D((2,2))(e3) # Уменьшаем размер (32x32 -> 16x16).

    # --- Мост (самая узкая часть) ---
    # Соединяет энкодер и декодер. Здесь изображение имеет наименьшее пространственное разрешение,
    # но наибольшее количество фильтров (наиболее абстрактное представление).
    bridge = residual_block(p3, 384) # Обрабатываем данные в узком месте.

    # --- Декодер (Расширяющая часть) ---
    # Декодер постепенно увеличивает пространственное разрешение изображения,
    # уменьшая количество фильтров, чтобы восстановить исходный размер и структуру.

    # Блок 1 декодера:
    # Conv2DTranspose (или "обратная свертка") используется для увеличения размера изображения.
    d0 = Conv2DTranspose(192, (3,3), strides=2, padding='same', kernel_initializer='he_normal')(bridge) # Увеличиваем размер (16x16 -> 32x32).
    # Ключевой момент U-Net: Связь пропуска (Skip Connection).
    # Объединяем (concatenate) выход текущего слоя декодера с выходом соответствующего слоя энкодера (e3).
    # Это помогает декодеру "увидеть" более мелкие детали из энкодера, которые могли потеряться при сжатии.
    d0 = concatenate([d0, e3]) # Объединяем увеличенное изображение с признаками из e3.
    d0 = residual_block(d0, 192) # Обрабатываем объединенные данные.

    # Блок 2 декодера:
    d1 = Conv2DTranspose(96, (3,3), strides=2, padding='same', kernel_initializer='he_normal')(d0) # Увеличиваем размер (32x32 -> 64x64).
    # Связь пропуска с e2.
    d1 = concatenate([d1, e2]) # Объединяем с признаками из e2.
    d1 = residual_block(d1, 96) # Обрабатываем.

    # Блок 3 декодера:
    d2 = Conv2DTranspose(48, (3,3), strides=2, padding='same', kernel_initializer='he_normal')(d1) # Увеличиваем размер (64x64 -> 128x128).
    # Связь пропуска с e1.
    d2 = concatenate([d2, e1]) # Объединяем с признаками из e1.
    d2 = residual_block(d2, 48) # Обрабатываем.

    # --- Выходной слой ---
    # Финальная свертка для получения нужного количества каналов (3 для RGB)
    # и применения активации, которая определит диапазон выходных значений.
    x = Conv2D(48, (3,3), activation='relu', padding='same', kernel_initializer='he_normal')(d2) # Еще один слой обработки.
    # Финальный слой: 3 канала (RGB). Активация 'sigmoid' сжимает выход в диапазон [0, 1],
    # что соответствует нормализованным значениям пикселей.
    # dtype=tf.float32 указываем явно, чтобы избежать потенциальных проблем с mixed precision на выходе.
    outputs = Conv2D(3, (1,1), activation='sigmoid', padding='same', kernel_initializer='he_normal', dtype=tf.float32)(x)

    # Создаем модель, указывая, откуда начинается вход и чем заканчивается выход.
    model = Model(inputs, outputs)

    # --- Компиляция модели ---
    # Этот шаг конфигурирует модель для процесса обучения.
    # Определяет, как модель будет обновлять свои веса (оптимизатор),
    # какую функцию использовать для расчета ошибки (loss),
    # и какие метрики отслеживать в процессе (metrics).

    # Определяем оптимизатор (Adam - популярный выбор). Скорость обучения берем из глобальных параметров.
    optimizer = Adam(learning_rate=INITIAL_LR)

    # Компилируем модель:
    model.compile(
        optimizer=optimizer, # Используем наш оптимизатор.
        loss='mse', # Mean Squared Error (MSE) - Среднеквадратичная ошибка.
        # Пытаемся минимизировать квадрат разницы между предсказанным и чистым изображением.
        metrics=[psnr_metric, ssim_metric] # Отслеживаем PSNR и SSIM в процессе обучения и оценки.
    )

    return model # Возвращаем готовую к обучению (или загрузке весов) модель.

In [ ]:
# Функция для пошагового обучения
def train_model_stepwise():
    print("Подготовка данных для пошагового обучения...")
    train_image_dir = './watermarked/wm-nowm/train/no-watermark'
    valid_image_dir = './watermarked/wm-nowm/valid/no-watermark'

    if not os.path.exists(train_image_dir) or not os.listdir(train_image_dir):
         print(f"Ошибка: Директория тренировочных данных не найдена или пуста: {train_image_dir}"); return None
    if not os.path.exists(valid_image_dir) or not os.listdir(valid_image_dir):
         print(f"Ошибка: Директория валидационных данных не найдена или пуста: {valid_image_dir}"); return None

    all_train_filenames = os.listdir(train_image_dir)
    all_valid_filenames = os.listdir(valid_image_dir)

    current_train_sample_size = min(TRAIN_SAMPLE_SIZE, len(all_train_filenames))
    current_valid_sample_size = min(VALID_SAMPLE_SIZE, len(all_valid_filenames))

    local_monitor_sample_count = min(MONITOR_VALIDATION_SAMPLES, len(all_valid_filenames))

    if current_train_sample_size == 0:
         print("Ошибка: Нет доступных тренировочных файлов."); return None
    if current_valid_sample_size == 0:
         print("Ошибка: Нет доступных валидационных файлов."); return None
    if local_monitor_sample_count == 0 and len(all_valid_filenames) > 0:
         print("Предупреждение: Не задано количество валидационных примеров для мониторинга или их нет в наличии.")

    train_filenames_pool = random.sample(all_train_filenames, current_train_sample_size)
    valid_filenames_pool = random.sample(all_valid_filenames, current_valid_sample_size)

    monitor_filenames = random.sample(valid_filenames_pool, local_monitor_sample_count)
    print(f"Генерируем {len(monitor_filenames)} фиксированных валидационных примеров для мониторинга...")
    X_monitor, y_monitor_original = load_and_preprocess_chunk(monitor_filenames, valid_image_dir, IMG_SIZE, is_train=False, datagen=None)

    actual_monitor_samples_loaded = len(X_monitor)

    if actual_monitor_samples_loaded == 0 and local_monitor_sample_count > 0:
         print("Предупреждение: Не удалось загрузить валидационные примеры для мониторинга.")

    # Усиленная аугментация данных
    datagen = ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=10,
        zoom_range=0.1,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.8, 1.2],
        channel_shift_range=20,
        fill_mode='nearest'
    )


    print("Building boosted model (predicting clean image, NO SE blocks, more filters)...")
    model = build_model_no_se_predict_clean_boosted_filters() # Используем модель с увеличенной емкостью
    print("Model built successfully.")


    best_val_ssim = -1.0
    steps_since_last_improvement = 0
    learning_rate = model.optimizer.learning_rate.numpy()
    print(f"\nНачальная скорость обучения установлена на: {learning_rate:.6f}")

    chunk_size = current_train_sample_size // NUM_TRAINING_STEPS
    if chunk_size == 0:
        print(f"Ошибка: Размер тренировочной выборки ({current_train_sample_size}) недостаточен для {NUM_TRAINING_STEPS} шагов. Увеличьте TRAIN_SAMPLE_SIZE или уменьшите NUM_TRAINING_STEPS."); return None

    print(f"Размер каждого тренировочного чанка: {chunk_size}")
    print(f"Пошаговое обучение: {NUM_TRAINING_STEPS} шагов, по {EPOCHS_PER_STEP} эпох на каждом шаге.")
    print(f"Общее примерное количество эффективных эпох: {NUM_TRAINING_STEPS * EPOCHS_PER_STEP}")


    total_start_time = time.time()

    for step in range(NUM_TRAINING_STEPS):
        print(f"\n--- Шаг обучения {step+1}/{NUM_TRAINING_STEPS} ---")
        step_start_time = time.time()

        random.shuffle(train_filenames_pool)

        current_chunk_filenames = train_filenames_pool[:chunk_size]
        print(f"Загрузка и подготовка {len(current_chunk_filenames)} изображений для шага {step+1}...")
        X_train_chunk, y_train_chunk_original = load_and_preprocess_chunk(
            current_chunk_filenames,
            train_image_dir,
            IMG_SIZE,
            is_train=True,
            datagen=datagen
        )

        if len(X_train_chunk) == 0:
             print(f"Warning: Нет данных в тренировочном чанке для шага {step+1}. Пропускаем шаг."); continue


        print(f"Тренировка на {len(X_train_chunk)} примерах (форма {X_train_chunk.shape}) в течение {EPOCHS_PER_STEP} эпох...")

        try:
             history_step = model.fit(
                 X_train_chunk, y_train_chunk_original,
                 batch_size=BATCH_SIZE,
                 epochs=EPOCHS_PER_STEP,
                 verbose=0,
             )
        except Exception as e:
             print(f"\nОшибка при обучении чанка: {e}")
             print("Пропуск текущего шага из-за ошибки model.fit.")
             if "Resource exhausted" in str(e) or "OOM" in str(e):
                  print("Вероятно, не хватило видеопамяти (OOM). Попробуйте уменьшить BATCH_SIZE или количество фильтров в модели.")
             del X_train_chunk, y_train_chunk_original
             gc.collect()
             continue


        step_train_time = time.time() - step_start_time
        print(f"Шаг {step+1} тренировка завершена за {step_train_time:.1f} секунд.")

        if actual_monitor_samples_loaded > 0:
             print(f"Оценка и визуализация после шага {step+1} на фиксированных валидационных примерах ({actual_monitor_samples_loaded} примеров)...")

             predicted_clean_monitor = model.predict(X_monitor, verbose=0)


             eval_psnr = psnr_metric(y_monitor_original.astype(np.float32), predicted_clean_monitor.astype(np.float32)).numpy().item()
             eval_ssim = ssim_metric(y_monitor_original.astype(np.float32), predicted_clean_monitor.astype(np.float32)).numpy().item()

             eval_loss = tf.reduce_mean(tf.square(y_monitor_original.astype(np.float32) - predicted_clean_monitor.astype(np.float32))).numpy().item()


             print(f"Валидация после шага {step+1}: Loss (MSE on clean): {eval_loss:.6f} | PSNR: {eval_psnr:.2f} dB | SSIM: {eval_ssim:.4f}")

             current_val_ssim = eval_ssim

             if current_val_ssim > best_val_ssim:
                 best_val_ssim = current_val_ssim
                 steps_since_last_improvement = 0
                 print(f"Валидация SSIM улучшилась до {best_val_ssim:.4f}. Сохранение весов модели в {MODEL_SAVE_PATH}...")
                 try:
                     model.save_weights(MODEL_SAVE_PATH)
                 except Exception as e:
                     print(f"Ошибка при сохранении весов модели: {e}")
             else:
                 steps_since_last_improvement += 1
                 print(f"Валидация SSIM не улучшилась. Шагов без улучшения: {steps_since_last_improvement}")

             current_lr = model.optimizer.learning_rate.numpy()
             if steps_since_last_improvement >= LR_REDUCTION_PATIENCE_STEPS and current_lr > MIN_LR:
                 learning_rate = current_lr * LR_REDUCTION_FACTOR
                 learning_rate = max(learning_rate, MIN_LR)
                 model.optimizer.learning_rate.assign(learning_rate)
                 print(f"Валидация не улучшалась {LR_REDUCTION_PATIENCE_STEPS} шагов. Снижение скорости обучения до {learning_rate:.6f}")
                 steps_since_last_improvement = 0

             if steps_since_last_improvement >= EARLY_STOPPING_PATIENCE_STEPS:
                 print(f"Валидация не улучшалась {EARLY_STOPPING_PATIENCE_STEPS} шагов. Ранняя остановка обучения.")
                 if os.path.exists(MODEL_SAVE_PATH):
                     print(f"\nЗагрузка лучших сохраненных весов из {MODEL_SAVE_PATH} перед остановкой.")
                     try:
                         model.load_weights(MODEL_SAVE_PATH)
                     except Exception as e:
                         print(f"Ошибка при загрузке лучших весов перед остановкой: {e}")
                 break

             plt.figure(figsize=(20, 5 * actual_monitor_samples_loaded))
             plt.suptitle(f"Шаг обучения: {step+1}/{NUM_TRAINING_STEPS} - Примеры валидации (Предсказание чистого)", fontsize=16)

             for i in range(actual_monitor_samples_loaded):
                 watermarked_sample = X_monitor[i]
                 original_sample = y_monitor_original[i]
                 predicted_clean_sample = predicted_clean_monitor[i]

                 removed_component_sample = watermarked_sample.astype(np.float32) - predicted_clean_sample.astype(np.float32)
                 removed_component_sample_clipped = np.clip(removed_component_sample, -1, 1)


                 current_ssim_sample = ssim_metric(original_sample.astype(np.float32), predicted_clean_sample.astype(np.float32)).numpy().item()
                 current_psnr_sample = psnr_metric(original_sample.astype(np.float32), predicted_clean_sample.astype(np.float32)).numpy().item()


                 plt.subplot(actual_monitor_samples_loaded, 4, 4*i + 1)
                 plt.title("1. Вход (с WM)")
                 plt.imshow(np.clip(watermarked_sample, 0, 1))
                 plt.axis('off')

                 plt.subplot(actual_monitor_samples_loaded, 4, 4*i + 2)
                 plt.imshow(np.clip(predicted_clean_sample, 0, 1))
                 plt.title(f"2. Предсказание (Чистое)\nPSNR: {current_psnr_sample:.2f} dB, SSIM: {current_ssim_sample:.4f}")
                 plt.axis('off')

                 plt.subplot(actual_monitor_samples_loaded, 4, 4*i + 3)
                 plt.title("3. Оригинал (Чистое)")
                 plt.imshow(np.clip(original_sample, 0, 1))
                 plt.axis('off')

                 plt.subplot(actual_monitor_samples_loaded, 4, 4*i + 4)
                 plt.title("4. Удаленный компонент (Input - Pred)")

                 plt.imshow(removed_component_sample_clipped, cmap='gray', vmin=-1, vmax=1)
                 plt.axis('off')

             plt.tight_layout(rect=[0, 0.03, 1, 0.95])
             plt.show()
             print("-------------------------------------\n")

        elif local_monitor_sample_count > 0:
             print("Пропущена оценка и визуализация из-за ошибки загрузки валидационных примеров для мониторинга.")


        del X_train_chunk, y_train_chunk_original
        gc.collect()


    total_train_time = time.time() - total_start_time
    print(f"\nОбщее время пошагового обучения: {total_train_time/60:.1f} минут")
    mem_used = psutil.virtual_memory().used/(1024**3)
    print(f"Использование памяти после обучения: {mem_used:.1f} GB")

    if os.path.exists(MODEL_SAVE_PATH):
        print(f"\nЗагрузка лучших сохраненных весов из {MODEL_SAVE_PATH} для финальной оценки.")
        try:
            if model is not None:
                model.load_weights(MODEL_SAVE_PATH)
            else:
                print("Модель не была создана, загрузка весов невозможна.")
        except Exception as e:
            print(f"Ошибка при загрузке лучших весов перед оценкой: {e}")
            print("Используется модель с весами последнего шага.")
    else:
        print("\nЛучшие веса не были сохранены (возможно, не было улучшений валидации). Используется модель с весами последнего шага.")

    del X_monitor, y_monitor_original
    gc.collect()


    return model

In [ ]:
# Визуализация сравнения
def visualize_comparison(model, X_watermarked_test, y_original_test, n=10):
    """Visualizes comparison of watermarked input, predicted clean output, original clean, and removed component."""
    if len(X_watermarked_test) == 0 or len(y_original_test) == 0 or len(X_watermarked_test.shape) != y_original_test.shape:
        print("Нет данных для визуализации или данные не соответствуют."); return

    actual_n = min(n, len(X_watermarked_test))
    if actual_n == 0:
         print("Недостаточно примеров для визуализации."); return

    print(f"\nДемонстрация {actual_n} случайных примеров из тестовой выборки (с разнообразным WM)...")

    if actual_n > len(X_watermarked_test):
         indices = np.arange(len(X_watermarked_test))
    else:
         indices = np.random.choice(len(X_watermarked_test), actual_n, replace=False)


    predicted_clean_imgs = model.predict(X_watermarked_test[indices], verbose=0)

    plt.figure(figsize=(20, 5 * actual_n))
    plt.suptitle("Тестовые результаты: Вход (с WM), Предсказание (Чистое), Оригинал (Чистое), Удаленный компонент", fontsize=16)


    for i in range(actual_n):
        original_test_idx = indices[i]

        watermarked_sample = X_watermarked_test[original_test_idx]
        original_img = y_test_original[original_test_idx]
        predicted_clean_sample = predicted_clean_imgs[i]

        removed_component_sample = watermarked_sample.astype(np.float32) - predicted_clean_sample.astype(np.float32)

        removed_component_sample_clipped = np.clip(removed_component_sample, -1, 1)


        current_ssim_sample = ssim_metric(original_img.astype(np.float32), predicted_clean_sample.astype(np.float32)).numpy().item()
        current_psnr_sample = psnr_metric(original_img.astype(np.float32), predicted_clean_sample.astype(np.float32)).numpy().item()


        plt.subplot(actual_n, 4, 4*i + 1)
        plt.title("1. Вход (с WM)")

        plt.imshow(np.clip(watermarked_sample, 0, 1))
        plt.axis('off')

        plt.subplot(actual_n, 4, 4*i + 2)

        plt.imshow(np.clip(predicted_clean_sample, 0, 1))
        plt.title(f"2. Предсказание (Чистое)\nPSNR: {current_psnr_sample:.2f} dB, SSIM: {current_ssim_sample:.4f}")
        plt.axis('off')

        plt.subplot(actual_n, 4, 4*i + 3)
        plt.title("3. Оригинал (Чистое)")

        plt.imshow(np.clip(original_img, 0, 1))
        plt.axis('off')

        plt.subplot(actual_n, 4, 4*i + 4)
        plt.title("4. Удаленный компонент (Input - Pred)")

        plt.imshow(removed_component_sample_clipped, cmap='gray', vmin=-1, vmax=1)
        plt.axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


In [ ]:
# Запуск обучения и оценки
if __name__ == "__main__":
    print("Программа запущена.")
    print("Загрузка и распаковка датасета 'watermarked.zip'...")

    if not os.path.exists('./watermarked/wm-nowm/train/no-watermark'):
         print("Директория для тренировочных данных не найдена. Попытка скачать и распаковать датасет.")
         zip_path = os.path.join('.', 'watermarked.zip')
         extract_dir = os.path.join('.', 'watermarked')

         if os.name == 'posix':
             try:
                 if os.system('which wget > /dev/null') != 0:
                      print("wget не найден. Пожалуйста, установите wget или скачайте файл вручную.")
                 else:
                      os.system(f'wget -q https://storage.yandexcloud.net/academy.ai/watermarked.zip -O {zip_path}')
                      if not os.path.exists(zip_path):
                          print(f"Ошибка: файл {zip_path} не был загружен.")
                      else:
                          print(f"Файл {zip_path} загружен.")
                          if os.system('which unzip > /dev/null') != 0:
                               print("unzip не найден. Пожалуйста, установите unzip или распакуйте файл вручную.")
                          else:
                              os.system(f'unzip -qo {zip_path} -d {extract_dir}')
                              if os.path.exists('./watermarked/wm-nowm/train/no-watermark'):
                                   print("Датасет успешно загружен и распакован.")
                              else:
                                  print(f"Ошибка: Файл {zip_path} не содержит ожидаемой структуры папок после распаковки в {extract_dir}.")
             except Exception as e:
                 print(f"Ошибка при загрузке или распаковке датасета: {e}")
                 print("Пожалуйста, убедитесь, что файл watermarked.zip доступен и распакован в папку ./watermarked.")

         else:
             print("Автоматическая загрузка и распаковка поддерживается только для Linux/macOS.")
             print(f"Пожалуйста, скачайте датасет вручную по ссылке: https://storage.yandexcloud.net/academy.ai/watermarked.zip")
             print(f"И распакуйте его в директорию ./watermarked (должна появиться структура папок wm-nowm/train/no-watermark и т.д.)")


         if not os.path.exists('./watermarked/wm-nowm/train/no-watermark'):
              exit("Не удалось подготовить директории данных. Завершение работы.")


    print("Датасет должен быть готов в директории ./watermarked.")

    trained_model = train_model_stepwise()

    if trained_model is None:
        print("Обучение модели не удалось. Завершение работы.")
        exit()

    print("\n--- Оценка модели на тестовых данных ---")
    print("Загрузка тестовых данных для ФИНАЛЬНОЙ оценки (с РАЗНООБРАЗНЫМ знаком)...")

    test_data_base_dir = './watermarked/wm-nowm/valid/no-watermark'
    if not os.path.exists(test_data_base_dir) or not os.listdir(test_data_base_dir):
        print(f"ПРЕДУПРЕЖДЕНИЕ: Валидационные данные для теста не найдены в {test_data_base_dir}. Попробуем использовать тренировочные.")
        test_data_base_dir = './watermarked/wm-nowm/train/no-watermark'
        if not os.path.exists(test_data_base_dir) or not os.listdir(test_data_base_dir):
            print(f"КРИТИЧЕСКАЯ ОШИБКА: Данные для теста не найдены. Проверьте датасет."); exit();

    all_test_filenames = os.listdir(test_data_base_dir)
    current_test_sample_size = min(TEST_SAMPLE_SIZE, len(all_test_filenames))

    if current_test_sample_size == 0:
        print("КРИТИЧЕСКАЯ ОШИБКА: В директории для тестирования нет файлов. Завершение работы."); exit();

    test_filenames = random.sample(all_test_filenames, current_test_sample_size)
    print(f"Загрузка {len(test_filenames)} тестовых изображений (оригиналы) из {test_data_base_dir}...Изображения будут уменьшены до {IMG_SIZE}")
    _, clean_images_test = load_and_preprocess_chunk(test_filenames, test_data_base_dir, IMG_SIZE, is_train=False, datagen=None)


    if len(clean_images_test) == 0:
        print("КРИТИЧЕСКАЯ ОШИБКА: Не удалось загрузить тестовые изображения. Завершение работы."); exit();


    print(f"Применение разнообразного водяного знака к {len(clean_images_test)} тестовым изображениям...")
    X_test_watermarked = np.array([add_watermark(img, is_debug_simple=False) for img in clean_images_test])
    y_test_original = clean_images_test.copy()

    del clean_images_test
    gc.collect()

    print(f"Тестовые данные (с разнообразным знаком): X_test форма: {X_test_watermarked.shape}, y_test форма: {y_test_original.shape}")

    if len(X_test_watermarked) > 0:
        print("\nИтоговые метрики на тестовых данных (с РАЗНООБРАЗНЫМ знаком):")

        predicted_clean_imgs_test = trained_model.predict(X_test_watermarked, verbose=1)

        final_psnr = psnr_metric(y_test_original.astype(np.float32), predicted_clean_imgs_test.astype(np.float32)).numpy().item()
        final_ssim = ssim_metric(y_test_original.astype(np.float32), predicted_clean_imgs_test.astype(np.float32)).numpy().item()


        print(f"PSNR: {final_psnr:.2f} dB | SSIM: {final_ssim:.4f}")


        visualize_comparison(trained_model, X_test_watermarked, y_test_original, n=10)

    else:
        print("Тестовые данные не подготовлены для оценки.")

    print("\nПрограмма завершена.")

In [ ]:
import numpy as np
import cv2
import os
import tensorflow as tf
import random
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, BatchNormalization, PReLU, Add
from tensorflow.keras.models import Model

IMG_WIDTH = 128
IMG_HEIGHT = 128
IMG_SIZE = (IMG_WIDTH, IMG_HEIGHT)

import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings(
    'ignore',
    category=UserWarning,
    module='matplotlib.image',
    message='Clipping input data.*'
)

def residual_block(x, filters):
    shortcut = x
    if shortcut.shape[-1] is not None and shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1,1), padding='same', kernel_initializer='he_normal', dtype=x.dtype)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    x = Conv2D(filters, (3,3), padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = PReLU(shared_axes=[1,2])(x)
    x = Conv2D(filters, (3,3), padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Add()([shortcut, x])
    x = PReLU(shared_axes=[1,2])(x)
    return x

def build_model_no_se_predict_clean_boosted_filters():
    print("Используется архитектура модели из вашего скрипта. Убедитесь, что она верна.")
    inputs = Input((*IMG_SIZE, 3), dtype=tf.float32)
    e1 = residual_block(inputs, 48)
    p1 = MaxPooling2D((2,2))(e1)
    e2 = residual_block(p1, 96)
    p2 = MaxPooling2D((2,2))(e2)
    e3 = residual_block(p2, 192)
    p3 = MaxPooling2D((2,2))(e3)
    bridge = residual_block(p3, 384)
    d0 = Conv2DTranspose(192, (3,3), strides=2, padding='same', kernel_initializer='he_normal')(bridge)
    d0 = concatenate([d0, e3])
    d0 = residual_block(d0, 192)
    d1 = Conv2DTranspose(96, (3,3), strides=2, padding='same', kernel_initializer='he_normal')(d0)
    d1 = concatenate([d1, e2])
    d1 = residual_block(d1, 96)
    d2 = Conv2DTranspose(48, (3,3), strides=2, padding='same', kernel_initializer='he_normal')(d1)
    d2 = concatenate([d2, e1])
    d2 = residual_block(d2, 48)
    x = Conv2D(48, (3,3), activation='relu', padding='same', kernel_initializer='he_normal')(d2)
    outputs = Conv2D(3, (1,1), activation='sigmoid', padding='same', kernel_initializer='he_normal', dtype=tf.float32)(x)
    model = Model(inputs, outputs)
    return model

# ==============================================================================
# === Основная функция для обработки выборки изображений и визуализации =========
# ==============================================================================

def process_images_and_visualize_output(model_weights_path, watermarked_image_dir, num_images_to_test):
    """
    Загружает веса обученной модели, обрабатывает случайную выборку изображений
    из папки с WM и отображает входное и выходное изображения для каждого.

    Args:
        model_weights_path (str): Путь к сохраненному файлу весов модели (.weights.h5).
        watermarked_image_dir (str): Путь к папке с изображениями для тестирования (с WM).
        num_images_to_test (int): Количество изображений для случайной выборки из папки.
    """
    # Проверка существования файла весов модели
    if not os.path.exists(model_weights_path):
        print(f"Ошибка: Файл весов модели не найден по пути: {model_weights_path}")
        return

    if not os.path.exists(watermarked_image_dir):
        print(f"Ошибка: Папка с изображениями (с WM) не найдена по пути: {watermarked_image_dir}")
        return

    all_wm_filenames = os.listdir(watermarked_image_dir)
    wm_image_filenames = [f for f in all_wm_filenames if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]

    if not wm_image_filenames:
        print(f"Ошибка: Входная папка '{watermarked_image_dir}' не содержит поддерживаемых файлов изображений.")
        return

    num_available_images = len(wm_image_filenames)
    num_to_process = min(num_images_to_test, num_available_images)

    if num_to_process == 0:
        print(f"Внимание: Нет доступных изображений для обработки в папке '{watermarked_image_dir}' после фильтрации.")
        return

    selected_wm_filenames = random.sample(wm_image_filenames, num_to_process)

    # --- 1. Построение архитектуры модели ---
    print("Построение архитектуры модели...")
    try:
        model = build_model_no_se_predict_clean_boosted_filters()
        print("Архитектура модели успешно построена.")
    except Exception as e:
        print(f"Ошибка при построении архитектуры модели: {e}")
        return

    # --- 2. Загрузка весов в модель ---
    print(f"Загрузка весов из {model_weights_path}...")
    try:
        model.load_weights(model_weights_path)
        print("Веса модели успешно загружены.")
    except Exception as e:
        print(f"Ошибка при загрузке весов модели: {e}")
        print("Убедитесь, что архитектура модели в тестовом скрипте точно совпадает с архитектурой,")
        print("использованной при сохранении весов, и что файл весов не поврежден.")
        return

    # --- 3. Обработка выбранных изображений и визуализация ---
    print(f"\nНачало обработки случайной выборки из {num_to_process} изображений и визуализации...")

    processed_count = 0

    for filename in selected_wm_filenames:
        wm_image_path = os.path.join(watermarked_image_dir, filename)

        try:
            img_wm = cv2.imread(wm_image_path)
            if img_wm is None:
                 print(f"\nПредупреждение: Не удалось прочитать файл WM изображения: {wm_image_path}. Пропуск.")
                 continue
            if img_wm.ndim == 2: img_wm = cv2.cvtColor(img_wm, cv2.COLOR_GRAY2RGB)
            elif img_wm.shape[2] == 4: img_wm = cv2.cvtColor(img_wm, cv2.COLOR_BGRA2BGR)
            img_wm = cv2.resize(img_wm, IMG_SIZE)
            img_wm = cv2.cvtColor(img_wm, cv2.COLOR_BGR2RGB) # Ensure RGB
            input_image_for_model = img_wm.astype(np.float32) / 255.0 # Нормализация для модели [0, 1]
            input_image_for_model_batch = np.expand_dims(input_image_for_model, axis=0) # Форма (1, 128, 128, 3)


            # --- Выполнение предсказания моделью ---
            predicted_image_batch = model.predict(input_image_for_model_batch, verbose=0)
            predicted_image_normalized = predicted_image_batch[0]

            # --- Постобработка выходного изображения ---
            predicted_image_np = predicted_image_normalized * 255.0
            predicted_image_np = np.clip(predicted_image_np, 0, 255).astype(np.uint8) # Денормализация и клиппинг [0, 255]


            # --- Визуализация ---
            plt.figure(figsize=(12, 6))
            plt.suptitle(f"Файл: {filename}", fontsize=12)

            plt.subplot(1, 2, 1)
            plt.title("1. Вход (с WM)")
            # Для отображения нормализованного входа
            plt.imshow(np.clip(input_image_for_model, 0, 1))
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.title("2. Результат модели (без WM)")
            # Для отображения uint8 результата
            plt.imshow(predicted_image_np)
            plt.axis('off')

            plt.tight_layout()
            plt.show()

            processed_count += 1

        except Exception as e:
             print(f"Ошибка при обработке или визуализации файла {filename}: {e}. Пропуск.")
             continue

    print(f"\nОбработка и визуализация {processed_count} изображений завершена.")


# ==============================================================================
# === ГЛАВНЫЙ БЛОК ДЛЯ ЗАПУСКА СКРИПТА ТЕСТИРОВАНИЯ ==============================
# ==============================================================================

if __name__ == "__main__":
    print("Запуск скрипта тестирования модели (визуализация выборки)...")

    model_weights_file = 'best_denoiser_model_weights.weights.h5'

    watermarked_images_directory = '/content/watermarked/wm-nowm/valid/watermark'

    num_images_to_process = 10
    if not watermarked_images_directory or not os.path.exists(watermarked_images_directory):
        print("\nОшибка: Путь к папке с WM изображениями не указан или папка не найдена.")
        print(f"Переменная 'watermarked_images_directory' сейчас равна: '{watermarked_images_directory}'")
    else:
        process_images_and_visualize_output(
            model_weights_file,
            watermarked_images_directory,
            num_images_to_process
        )

    print("Скрипт тестирования завершен.")

В рамках этого проекта мы разработали модель на основе архитектуры типа U-Net для удаления текстовых водяных знаков с изображений. Использовали предоставленный набор чистых картинок, на который сами наносили разнообразные текстовые водяные знаки для создания обучающих пар. Чтобы эффективно использовать память и улучшить результат, применили расширенную аугментацию данных и реализовали специальный пошаговый процесс обучения: загружали данные частями, обучали модель, чистили память и динамически подстраивали скорость обучения, ориентируясь на качество предсказаний на отдельной валидационной выборке. Тестирование показало, что модель научилась вносить измеримые изменения в изображения, приближая их к чистым оригиналам на данных, похожих на те, что использовались при обучении (это видно по метрикам PSNR и SSIM на валидационных данных, где PSNR достигает ~32 dB, а SSIM ~0.87+). Однако, на совсем новых картинках, которые модель не видела, она пока работает неэффективно или вообще не меняет изображение. Система включает полный пайплайн для обучения и тестирования, а также подготовку вывода изображений (оригинал, с WM, предсказанное, шум) для анализа результата.